# Early Production: LangChain + HuggingFace
*Structured Applications with Local Models*

This notebook demonstrates how to build production-ready applications using LangChain with HuggingFace models. Perfect for early production when you need structured workflows but want to keep models local.

## When to Use This Approach
- Building structured LLM applications
- Need chains, agents, or memory management
- Want error handling and logging
- Planning to scale to multiple providers later
- Early production with local model requirements

## Prerequisites
- langchain
- langchain-huggingface
- transformers
- torch

## Setup and Environment

In [25]:
# Install required packages (run in terminal if needed)
# pip install langchain langchain-huggingface transformers torch

import torch
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Basic LangChain + HuggingFace Integration

In [26]:
# Create HuggingFace pipeline through LangChain
def create_hf_llm(model_name="gpt2", task="text-generation", **kwargs):
    """
    Create a LangChain-compatible LLM from HuggingFace model
    """
    # Base pipeline configuration
    pipeline_kwargs = {
        "max_new_tokens": 150,
        "temperature": 0.7,
        "do_sample": True,
        "pad_token_id": None,  # Will be set automatically
        **kwargs
    }
    
    llm = HuggingFacePipeline.from_model_id(
        model_id=model_name,
        task=task,
        pipeline_kwargs=pipeline_kwargs,
        device=0 if torch.cuda.is_available() else -1
    )
    
    return llm

# Create LLM instance
llm = create_hf_llm("gpt2")
print(f"LLM created: {llm}")
print(f"Model: {llm.pipeline.model.config.name_or_path}")

In [12]:
# Basic inference with LangChain
def basic_inference_demo():
    """
    Demonstrate basic inference patterns
    """
    print("🧪 Basic Inference Demo")
    
    # Simple generation
    prompt = "The future of renewable energy is"
    response = llm.invoke(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()
    
    # Multiple prompts
    prompts = [
        "Explain quantum computing in simple terms:",
        "Write a haiku about artificial intelligence:",
        "What are the benefits of exercise?"
    ]
    
    for prompt in prompts:
        response = llm.invoke(prompt)
        print(f"Q: {prompt}")
        print(f"A: {response[:100]}...")
        print()

basic_inference_demo()

## Structured Chains and Templates

In [27]:
# Create structured prompt templates
def create_structured_chains():
    """
    Demonstrate structured chains with templates
    """
    print("🔗 Structured Chains Demo")
    
    # Template for creative writing
    creative_template = PromptTemplate(
        input_variables=["topic", "style"],
        template="""Write a short {style} story about {topic}.
        
        Story:"""
    )
    
    creative_chain = creative_template | llm
    
    # Template for analysis
    analysis_template = PromptTemplate(
        input_variables=["subject", "aspect"],
        template="""Analyze the {aspect} of {subject}.
        
        Provide a structured analysis with:
        1. Overview
        2. Key points
        3. Conclusion
        
        Analysis:"""
    )
    
    analysis_chain = analysis_template | llm
    
    # Test creative chain
    creative_result = creative_chain.invoke({"topic": "time travel", "style": "sci-fi"})
    print("🎨 Creative Writing Chain:")
    print(f"Topic: time travel, Style: sci-fi")
    print(f"Result: {creative_result[:200]}...")
    print()
    
    # Test analysis chain
    analysis_result = analysis_chain.invoke({"subject": "machine learning", "aspect": "impact on healthcare"})
    print("📊 Analysis Chain:")
    print(f"Subject: machine learning, Aspect: healthcare impact")
    print(f"Result: {analysis_result[:200]}...")
    print()

create_structured_chains()

## Conversation Memory and State Management

In [17]:
# Conversation with message history
def conversation_demo():
    """
    Demonstrate conversation with message history
    """
    print("💬 Conversation with Message History Demo")
    
    # Create a simple conversation chain using RunnableWithMessageHistory
    def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
        return InMemoryChatMessageHistory()
    
    # Create a basic conversation runnable
    conversation_prompt = PromptTemplate(
        input_variables=["input", "history"],
        template="""You are a helpful AI assistant. Use the conversation history to continue naturally.

History:
{history}

Current input: {input}

Response:"""
    )
    
    conversation_chain = conversation_prompt | llm
    
    # Wrap with message history
    conversation_with_history = RunnableWithMessageHistory(
        conversation_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    
    # Simulate conversation
    messages = [
        "Hello! My name is Alex.",
        "I'm interested in learning about AI. Can you help me?",
        "What's the difference between machine learning and deep learning?",
        "That makes sense. What programming language should I learn first?"
    ]
    
    session_id = "demo_session"
    for message in messages:
        response = conversation_with_history.invoke(
            {"input": message},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"User: {message}")
        print(f"AI: {response[:100]}...")
        print()
    
    # Show history contents
    history = get_session_history(session_id)
    print("🧠 Message History:")
    for msg in history.messages:
        print(f"{msg.type}: {msg.content[:50]}...")
    print()

conversation_demo()

## Error Handling and Robustness

In [18]:
# Production-ready error handling
def robust_chain_execution(chain, inputs, max_retries=3):
    """
    Execute chain with error handling and retries
    """
    for attempt in range(max_retries):
        try:
            result = chain.invoke(inputs)
            return result, None
        except Exception as e:
            error_msg = f"Attempt {attempt + 1} failed: {str(e)}"
            print(error_msg)
            
            if attempt == max_retries - 1:
                return None, error_msg
            
            # Wait before retry (exponential backoff)
            import time
            time.sleep(2 ** attempt)
    
    return None, "Max retries exceeded"

def error_handling_demo():
    """
    Demonstrate error handling in production scenarios
    """
    print("🛡️ Error Handling Demo")
    
    # Create a chain that might fail
    template = PromptTemplate(
        input_variables=["topic"],
        template="Write a detailed analysis of {topic} in exactly 500 words."
    )
    
    chain = template | llm
    
    # Test with various inputs
    test_cases = [
        {"topic": "artificial intelligence"},
        {"topic": "quantum physics"},
        {"topic": "sustainable energy"}
    ]
    
    for i, inputs in enumerate(test_cases, 1):
        print(f"\nTest Case {i}: {inputs['topic']}")
        result, error = robust_chain_execution(chain, inputs, max_retries=2)
        
        if result:
            print(f"✅ Success: {result[:150]}...")
        else:
            print(f"❌ Failed: {error}")
    
    print("\n💡 Error handling ensures production reliability!")

error_handling_demo()

## Multi-Model Comparison and Fallback

In [19]:
# Multi-model setup for robustness
def create_multi_model_setup():
    """
    Create setup with multiple models for fallback
    """
    print("🔄 Multi-Model Setup Demo")
    
    models = {
        "gpt2": create_hf_llm("gpt2", max_length=50),
        "distilgpt2": create_hf_llm("distilgpt2", max_length=50)
    }
    
    def robust_generation(prompt, preferred_model="gpt2"):
        """
        Generate with fallback to alternative models
        """
        for model_name in [preferred_model] + [m for m in models.keys() if m != preferred_model]:
            try:
                llm_instance = models[model_name]
                result = llm_instance.invoke(prompt)
                return result, model_name
            except Exception as e:
                print(f"Model {model_name} failed: {e}")
                continue
        
        return "All models failed", None
    
    # Test multi-model setup
    test_prompts = [
        "Explain neural networks simply",
        "What is the capital of France?",
        "Describe a sunset"
    ]
    
    for prompt in test_prompts:
        result, model_used = robust_generation(prompt)
        print(f"Prompt: {prompt}")
        print(f"Model: {model_used}")
        print(f"Result: {result[:100]}...")
        print()

create_multi_model_setup()

## Logging and Monitoring

In [20]:
# Basic logging and monitoring
import logging
import time
from functools import wraps

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def log_execution(func):
    """
    Decorator to log function execution
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        logger.info(f"Starting {func.__name__}")
        
        try:
            result = func(*args, **kwargs)
            execution_time = time.time() - start_time
            logger.info(".3f")
            return result
        except Exception as e:
            execution_time = time.time() - start_time
            logger.error(".3f")
            raise
    
    return wrapper

@log_execution
def monitored_generation(prompt):
    """
    Monitored generation function
    """
    return llm.invoke(prompt)

def monitoring_demo():
    """
    Demonstrate logging and monitoring
    """
    print("📊 Monitoring Demo")
    
    test_prompts = [
        "Hello world",
        "What is AI?",
        "Tell me a joke"
    ]
    
    print("Check the logs above for execution details:")
    for prompt in test_prompts:
        result = monitored_generation(prompt)
        print(f"Prompt: {prompt}")
        print(f"Result: {result[:50]}...")
        print()

monitoring_demo()

## 🚀 Early Production Best Practices

ARCHITECTURE:
- Use LangChain for structured application logic  
- Implement proper error handling and retries  
- Add logging and monitoring from day one  
- Design for easy provider switching later  

RELIABILITY:
- Implement fallback mechanisms  
- Add input validation and sanitization  
- Use conversation memory appropriately  
- Handle rate limits and API constraints  

PERFORMANCE:
- Profile and optimize chain execution  
- Use appropriate model sizes for your needs  
- Implement caching for repeated operations  
- Monitor memory usage and latency  

MAINTAINABILITY:
- Use clear prompt templates and chain names  
- Document chain purposes and inputs  
- Implement proper configuration management  
- Write tests for critical chains